In [1]:
import pandas as pd

fhv_df = pd.read_csv("fhvhv_tripdata_2025-08.csv")


In [2]:
fhv_df = fhv_df[['hvfhs_license_num', 'request_datetime', 'pickup_datetime','dropoff_datetime', 'PULocationID', 'DOLocationID', 'wav_request_flag', 'wav_match_flag', 'driver_pay']]

In [3]:
fhv_df.isna().sum()

hvfhs_license_num    0
request_datetime     0
pickup_datetime      0
dropoff_datetime     0
PULocationID         0
DOLocationID         0
wav_request_flag     0
wav_match_flag       0
driver_pay           0
dtype: int64

In [4]:

fhv_df["request_datetime"] = pd.to_datetime(fhv_df["request_datetime"], errors="coerce")
fhv_df["pickup_datetime"] = pd.to_datetime(fhv_df["pickup_datetime"], errors="coerce")
fhv_df["dropoff_datetime"] = pd.to_datetime(fhv_df["dropoff_datetime"], errors="coerce")

fhv_df[['request_datetime', 'pickup_datetime', 'dropoff_datetime']].isna().sum()

request_datetime    0
pickup_datetime     0
dropoff_datetime    0
dtype: int64

In [7]:

invalid_time = fhv_df[(fhv_df["pickup_datetime"] <= fhv_df["request_datetime"]) | (fhv_df["dropoff_datetime"] <= fhv_df["pickup_datetime"])]
len(invalid_time)  

224540

In [8]:
fhv_df = fhv_df[(fhv_df["pickup_datetime"] > fhv_df["request_datetime"]) & (fhv_df["dropoff_datetime"] > fhv_df["pickup_datetime"])]

In [11]:
fhv_df["trip_minutes"] = (
    (fhv_df["dropoff_datetime"] - fhv_df["pickup_datetime"]).dt.total_seconds() / 60
)



In [12]:
fhv_df = fhv_df[(fhv_df["trip_minutes"] >= 5) & (fhv_df["trip_minutes"] < 180)]

In [13]:
fhv_df = fhv_df[fhv_df["driver_pay"] > 0]

In [14]:
invalid_loc = fhv_df[
    (fhv_df["PULocationID"] <= 0) |
    (fhv_df["DOLocationID"] <= 0) |
    (fhv_df["PULocationID"] > 263) |
    (fhv_df["DOLocationID"] > 263)
]
len(invalid_loc), invalid_loc.head()

(898561,
    hvfhs_license_num    request_datetime     pickup_datetime  \
 24            HV0003 2025-08-01 00:05:07 2025-08-01 00:13:19   
 30            HV0005 2025-07-31 23:58:03 2025-08-01 00:04:06   
 67            HV0003 2025-08-01 00:34:13 2025-08-01 00:38:56   
 79            HV0003 2025-08-01 00:13:55 2025-08-01 00:19:44   
 88            HV0003 2025-08-01 00:48:06 2025-08-01 00:50:54   
 
       dropoff_datetime  PULocationID  DOLocationID wav_request_flag  \
 24 2025-08-01 00:33:46           186           265                N   
 30 2025-08-01 00:34:08           186           265                N   
 67 2025-08-01 01:07:35           138           265                N   
 79 2025-08-01 01:05:33           163           265                N   
 88 2025-08-01 01:13:34           137           265                N   
 
    wav_match_flag  driver_pay  trip_minutes  
 24              N       34.04     20.450000  
 30              N       29.79     30.033333  
 67              N      

In [15]:
fhv_df = fhv_df[(fhv_df['PULocationID']<=263) & (fhv_df['PULocationID']>=1)]
fhv_df = fhv_df[(fhv_df['DOLocationID']<=263) & (fhv_df['DOLocationID']>=1)]

In [16]:
fhv_df.columns

Index(['hvfhs_license_num', 'request_datetime', 'pickup_datetime',
       'dropoff_datetime', 'PULocationID', 'DOLocationID', 'wav_request_flag',
       'wav_match_flag', 'driver_pay', 'trip_minutes'],
      dtype='object')

In [17]:
fhv_df = fhv_df.rename(columns={'hvfhs_license_num': 'service_provider', 'PULocationID': 'pickup_location', 'DOLocationID': 'dropoff_location', 'driver_pay': 'total_amount'})

In [18]:
fhv_df = fhv_df.drop(columns=["trip_minutes"])

In [19]:
fhv_df.to_csv('fhv_cleaned_df.csv')